# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:

print("GEMINI_API_KEY:", "SET" if os.getenv("GEMINI_API_KEY") else "NOT SET")
print("GROQ_API_KEY:", "SET" if os.getenv("GROQ_API_KEY") else "NOT SET")

In [ ]:


# --- Configuración de clientes ---

# Cliente Gemini
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_api_key = os.getenv("GEMINI_API_KEY")
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=gemini_api_key)

# Cliente Groq
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
groq_api_key = os.getenv("GROQ_API_KEY")
groq = OpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

# Cliente Ollama (local)
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")



In [ ]:
def ask(messages, provider="ollama", response_format=None, max_retries=3):
    kwargs = {"messages": messages}
    if response_format:
        kwargs["response_format"] = response_format

    last_error = None
    for attempt in range(max_retries):
        try:
            if provider == "gemini":
                response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", **kwargs)
            elif provider == "groq":
                response = groq.chat.completions.create(model="openai/gpt-oss-20b", **kwargs)
            else:
                response = ollama.chat.completions.create(model="llama3.2:3b", **kwargs)
            return response.choices[0].message.content
        except Exception as e:
            error_str = str(e)
            if "503" in error_str and provider == "gemini" and attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"Gemini overloaded. Retrying in {wait_time}s...")
                time.sleep(wait_time)
                last_error = e
                continue
            raise
    if last_error:
        raise last_error

In [ ]:
links = fetch_website_links("https://edwarddonner.com")
links

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [ ]:

link_system_prompt = """You are provided with a list of links found on a webpage.
You must decide which links are relevant for a company brochure (About, Company, Careers/Jobs pages).

CRITICAL FORMAT RULES:
- Respond with ONLY valid JSON, nothing else (no explanation, no markdown).
- Each link must have exactly two fields: "type" (a short label) and "url" (a single string).
- The "url" field must ALWAYS be a single string, NEVER a list.
- If there are multiple links of the same type (e.g. multiple social media links), 
  create a SEPARATE entry for each one — do not group them into a list.
- Do not include Terms of Service, Privacy Policy, or mailto: links.

Example of CORRECT format:
{
    "links": [
        {"type": "about page", "url": "https://company.com/about"},
        {"type": "careers page", "url": "https://company.com/careers"},
        {"type": "twitter", "url": "https://twitter.com/company"},
        {"type": "linkedin", "url": "https://linkedin.com/company/company"}
    ]
}

Example of INCORRECT format (do not do this):
{
    "links": [
        {"type": "social media", "url": ["https://twitter.com/company", "https://linkedin.com/company"]}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))

In [ ]:
def select_relevant_links(url, provider="ollama"):
    messages = [
        {"role": "system", "content": link_system_prompt},
        {"role": "user", "content": get_links_user_prompt(url)}
    ]
    result = ask(messages, provider=provider, response_format={"type": "json_object"})
    return json.loads(result)

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://edwarddonner.com",provider="gemini")

In [ ]:
select_relevant_links("https://edwarddonner.com",provider="groq")

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [ ]:
from urllib.parse import urlparse

def is_valid_url(url):
    try:
        result = urlparse(url)
        return all([result.scheme in ("http", "https"), result.netloc]) and " " not in url
    except:
        return False

In [ ]:
from urllib.parse import urljoin

def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    seen_urls = {url}  # including landing pages
    for link in relevant_links['links']:
        link_url = urljoin(url, link["url"].strip())  # solving issue with URL base
        if not is_valid_url(link_url):
            print(f"Skipping invalid URL: {link_url}")
            continue
        if link_url in seen_urls:
            print(f"Skipping duplicate URL: {link_url}")
            continue
        seen_urls.add(link_url)
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link_url)
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs ONLY if you have the information.
CRITICAL: Do not invent or infer ANY specific factual details that are not explicitly stated 
in the provided content. This includes but is not limited to: founding dates, office locations, 
follower/user counts, specific URLs, email addresses, phone numbers, physical addresses, 
social media handles, or statistics of any kind. If you don't have this information, either 
omit it entirely or state generally that the company offers X without specific unverified numbers.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [ ]:


def create_brochure(company_name, url, provider="gemini"):
    messages = [
        {"role": "system", "content": brochure_system_prompt},
        {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
    ]
    result = ask(messages, provider=provider)
    display(Markdown(result))
    return result

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co",provider="groq")

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url, provider="gemini"):
    messages = [
        {"role": "system", "content": brochure_system_prompt},
        {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
    ]

    if provider == "gemini":
        client, model = gemini, "gemini-3.1-flash-lite"
    elif provider == "groq":
        client, model = groq, "openai/gpt-oss-20b"
    else:
        client, model = ollama, "llama3.2:3b"

    stream = client.chat.completions.create(model=model, messages=messages, stream=True)

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
    return response

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>